# Imports

In [1]:
!wb resource mount

Successfully mounted workspace bucket resources.


In [2]:
import os
import re
import pandas as pd
import numpy as np

try:
    from google.cloud import bigquery
except ImportError:
    bigquery = None

import subprocess

cmd = """
source /home/jupyter/load-env.sh >/dev/null
env
"""

result = subprocess.run(
    ["bash", "-lc", cmd],
    capture_output=True,
    text=True,
    check=True,
)

for line in result.stdout.splitlines():
    if "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value

print("WORKSPACE_CDR:", os.environ.get("WORKSPACE_CDR"))
print("WORKSPACE_BUCKET:", os.environ.get("WORKSPACE_BUCKET"))

bucket = os.getenv("WORKSPACE_BUCKET")
cdr = os.environ.get("WORKSPACE_CDR")

if cdr is None:
    raise EnvironmentError(
        "WORKSPACE_CDR is not set. This script should be run inside an All of Us workspace."
    )

use_bqstorage = ("BIGQUERY_STORAGE_API_ENABLED" in os.environ)

WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R8
WORKSPACE_BUCKET: gs://cloned-mybucket-wb-rapid-artichoke-5786


# Load Model

In [3]:
!pip install transformers
!pip install torch==2.6 torchaudio==2.6 torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.1/794.1 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.0 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2024.11.6
    Uninstalling regex-2024.11.6:
      Successfully uninstalled regex-2024.11.6
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.7/766.7 MB 21.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 42.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import login

hf_token = os.environ["HF_TOKEN"]
login(hf_token)

tokenizer = AutoTokenizer.from_pretrained("medicalai/ClinicalBERT")
model = AutoModel.from_pretrained("medicalai/ClinicalBERT")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
print(tokenizer.get_input_embeddings())

AttributeError: BertTokenizer has no attribute get_input_embeddings

In [10]:
df_cnc = pd.read_csv('/home/jupyter/workspace/data_bucket/MedRep/concept_idx.csv')
df_cnc['concept_id'] = df_cnc['concept_id'].astype(str)
df_cnc = df_cnc.set_index('concept_id')
df_cnc_idx = pd.DataFrame(np.arange(119547,119547+df_cnc.shape[0]), index=df_cnc.index, columns=['idx'])
df_cnc_idx

/tmp/ipykernel_174/3250303433.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_cnc = pd.read_csv('/home/jupyter/workspace/data_bucket/MedRep/concept_idx.csv')


,idx
concept_id,
3529951,119547
3273491,119548
42487381,119549
4176390,119550
40423999,119551
...,...
4160711,7850283
4341366,7850284
36914614,7850285


In [15]:
df_concepts = pd.read_parquet('/home/jupyter/workspace/data_bucket/MEDS_DATA/MEDS_cohort/metadata/codes.parquet')
df_concepts

,code,description,parent_codes
